# 🧠 Mental Health Google Form — Research Analysis (v2 — Fixed)
### Ensemble: Your Model (LR/SVM/RF) + BART Zero-Shot

---
> **Run cells in order. Only edit the `⚙️ CONFIGURATION` cell.**

**15 Target Classes:**  
`addiction` · `adhd` · `alcoholism` · `anxiety` · `autism` · `bipolarreddit` · `bpd` · 
`depression` · `healthanxiety` · `lonely` · `mentalhealth` · `ptsd` · `schizophrenia` · 
`socialanxiety` · `suicidewatch`

### 🔴 Three Root Causes Fixed in This Version
1. **ZIP file**: `Mentalhealth.csv` is actually a ZIP — auto-extracted here
2. **Dual column layout**: Rows 1-7 had text in cols 35/36, rows 8+ in cols 2/3 — now both are read
3. **Thresholds too high**: 0.38/0.28 are for Reddit posts; survey language needs 0.20/0.12


In [ ]:
from pathlib import Path as _ProjectPath
import sys as _project_sys

_project_root = next(
    (
        p
        for p in (_ProjectPath.cwd(), *_ProjectPath.cwd().parents)
        if (p / "project_config.json").is_file()
    ),
    None,
)
if _project_root is None:
    raise FileNotFoundError(
        "Open VS Code in the organized project root before running this notebook."
    )
if str(_project_root / "tools") not in _project_sys.path:
    _project_sys.path.insert(0, str(_project_root / "tools"))
from project_paths import ProjectPaths as _ProjectPaths

_paths = _ProjectPaths(_project_root, 'google_form_part2_v2')

import os, warnings, re, zipfile

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
import joblib
import shap
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

plt.rcParams.update(
    {
        'figure.dpi': 120,
        'figure.facecolor': 'white',
        'axes.facecolor': '#f9f9f9',
        'axes.grid': True,
        'grid.alpha': 0.3,
        'font.family': 'DejaVu Sans',
    }
)
CLASS_NAMES = [
    'addiction',
    'adhd',
    'alcoholism',
    'anxiety',
    'autism',
    'bipolarreddit',
    'bpd',
    'depression',
    'healthanxiety',
    'lonely',
    'mentalhealth',
    'ptsd',
    'schizophrenia',
    'socialanxiety',
    'suicidewatch',
]
PALETTE = [
    '#e74c3c',
    '#3498db',
    '#2ecc71',
    '#f39c12',
    '#9b59b6',
    '#1abc9c',
    '#e67e22',
    '#34495e',
    '#e91e63',
    '#00bcd4',
    '#8bc34a',
    '#ff5722',
    '#607d8b',
    '#ffc107',
    '#795548',
]
CLASS_COLOR = {cls: PALETTE[i] for i, cls in enumerate(CLASS_NAMES)}
print('✅ Libraries loaded')
print(f'   SHAP {shap.__version__} | Pandas {pd.__version__} | NumPy {np.__version__}')


In [ ]:
CSV_FILE = _paths.input_path('Mentalhealth.csv')


Q1_COL_LAYOUT_A = None
Q2_COL_LAYOUT_A = None
Q1_COL_LAYOUT_B = None
Q2_COL_LAYOUT_B = None


CONFIDENCE_THRESHOLD = 0.20
BORDERLINE_LOWER = 0.12


REDDIT_WEIGHT = 0.20
BART_WEIGHT = 0.80


MODEL_DIR = _paths.classical_dir
TFIDF_FILE = 'tfidf_vectorizer.pkl'
LABEL_FILE = 'label_encoder.pkl'
LR_FILE = 'logistic_regression_model.pkl'
SVM_FILE = 'linear_svm_model.pkl'
RF_FILE = 'random_forest_model.pkl'

print('✅ Configuration set')
print(f'   CSV                : {CSV_FILE}')
print(f'   Condition threshold: {CONFIDENCE_THRESHOLD}  (survey-calibrated)')
print(f'   Borderline         : {BORDERLINE_LOWER}')
print(f'   Reddit / BART mix  : {REDDIT_WEIGHT:.0%} / {BART_WEIGHT:.0%}')


In [ ]:
print('🤖 Loading pre-trained models...')
print('=' * 70)

if not os.path.exists(os.path.join(MODEL_DIR, TFIDF_FILE)):
    MODEL_DIR = _paths.classical_dir

print(f'   📂 Loading from: {MODEL_DIR}')


def _load(filename):
    path = os.path.join(MODEL_DIR, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(f'❌ Cannot find: {path}')
    return joblib.load(_paths.input_path(path))


tfidf = _load(TFIDF_FILE)
print(f'   ✅ TF-IDF ({tfidf.max_features} features)')
le = _load(LABEL_FILE)
print(f'   ✅ Label encoder ({len(le.classes_)} classes)')
lr = _load(LR_FILE)
print('   ✅ Logistic Regression')
svm = _load(SVM_FILE)
print('   ✅ Linear SVM')
rf = _load(RF_FILE)
print('   ✅ Random Forest')

models = {'Logistic Regression': lr, 'Linear SVM': svm, 'Random Forest': rf}
print(f'\n🎯 {len(le.classes_)} classes: {list(le.classes_)}')


In [ ]:
print('📂 Loading CSV (ZIP-aware)...')
print('=' * 70)


csv_path = _paths.input_path(
    os.path.join(MODEL_DIR, CSV_FILE) if not os.path.isabs(CSV_FILE) else CSV_FILE
)


def _resolve_csv(path):
    """If path is a ZIP, extract the first .csv inside and return its path."""
    import zipfile

    if zipfile.is_zipfile(path):
        print(f'   ℹ️  Detected ZIP archive: {os.path.basename(path)}')
        extract_dir = _paths.output_path(
            os.path.join(os.path.dirname(path), '_extracted_csv')
        )
        os.makedirs(extract_dir, exist_ok=True)
        with zipfile.ZipFile(_paths.input_path(path), 'r') as z:
            csv_names = [n for n in z.namelist() if n.lower().endswith('.csv')]
            if not csv_names:
                raise ValueError('❌ No .csv file found inside the ZIP.')
            z.extract(csv_names[0], extract_dir)
            extracted = os.path.join(extract_dir, csv_names[0])
            print(f'   ✅ Extracted: {csv_names[0]}')
            return extracted
    return path


csv_path = _paths.input_path(_resolve_csv(csv_path))


raw_df = None
for enc in ['utf-8', 'utf-8-sig', 'latin-1', 'cp1252']:
    try:
        raw_df = pd.read_csv(
            _paths.input_path(csv_path),
            encoding=enc,
            on_bad_lines='skip',
            engine='python',
        )
        print(f'   ✅ Encoding "{enc}" worked')
        break
    except UnicodeDecodeError:
        print(f'   ⚠️  "{enc}" failed, trying next...')

if raw_df is None:
    raise ValueError('❌ Could not read CSV.')

print(f'\n✅ Loaded  →  {raw_df.shape[0]} rows × {raw_df.shape[1]} columns')
for i, col in enumerate(raw_df.columns):
    print(f'   [{i:2d}] {col[:90]}')
raw_df.head(3)


In [ ]:
print('🔍 Extracting text — dual column layout...')
print('=' * 70)


def find_col_keyword(df, keyword, fallback_index=None):
    if fallback_index is not None:
        return df.columns[fallback_index]
    matches = [c for c in df.columns if keyword.lower() in c.lower()]
    return matches[0] if matches else None


col_a_q1 = find_col_keyword(raw_df, 'heaviest', Q1_COL_LAYOUT_A)
col_a_q2 = find_col_keyword(raw_df, 'alone', Q2_COL_LAYOUT_A)


col_b_rating = raw_df.columns[34] if raw_df.shape[1] > 34 else None
col_b_q1 = raw_df.columns[Q1_COL_LAYOUT_B or 35] if raw_df.shape[1] > 35 else None
col_b_q2 = raw_df.columns[Q2_COL_LAYOUT_B or 36] if raw_df.shape[1] > 36 else None


likert_cols = [raw_df.columns[j] for j in range(4, 34) if raw_df.shape[1] > j]

print(
    f'   Layout A — Q1: {str(col_a_q1)[:60]}'
    if col_a_q1
    else '   Layout A Q1 not found'
)
print(
    f'   Layout A — Q2: {str(col_a_q2)[:60]}'
    if col_a_q2
    else '   Layout A Q2 not found'
)
print(f'   Layout B — self-rating col: {col_b_rating}')
print(f'   Layout B — Q1 col: {col_b_q1}')
print(f'   Layout B — Q2 col: {col_b_q2}')
print(f'   Likert cols: {len(likert_cols)} found')


def likert_to_text(row, likert_cols):
    """Convert Likert frequency ratings into a descriptive text sentence.
    E.g. if 10 of 30 ratings are 'Very Often' or 'Often', we infer distress."""
    vals = [str(row[c]).strip() for c in likert_cols if pd.notna(row[c])]
    if not vals:
        return ''
    freq_map = {'Never': 0, 'Rarely': 1, 'Sometimes': 2, 'Often': 3, 'Very Often': 4}
    scores = [freq_map.get(v, 2) for v in vals]
    avg = sum(scores) / len(scores)
    often_count = sum(1 for v in vals if v in ('Often', 'Very Often'))
    if avg >= 3.0 or often_count >= 10:
        return (
            'I often feel overwhelmed, anxious or low. '
            'Many things bother me frequently. I struggle with my emotions often.'
        )
    elif avg >= 2.0 or often_count >= 5:
        return (
            'Sometimes I feel stressed or anxious. '
            'Some things bother me from time to time.'
        )
    else:
        return 'I mostly feel okay. Things are generally manageable for me.'


def extract_row_text(row):
    """Extract the best text from a row regardless of which layout was used."""
    parts = []

    if col_a_q1 and pd.notna(row[col_a_q1]):
        parts.append(str(row[col_a_q1]).strip())
    if col_a_q2 and pd.notna(row[col_a_q2]):
        parts.append(str(row[col_a_q2]).strip())

    if col_b_rating and pd.notna(row[col_b_rating]):

        sr = str(row[col_b_rating]).strip()
        if sr not in ('nan', ''):
            parts.append(f'Self-reported mood: {sr}.')
    if col_b_q1 and pd.notna(row[col_b_q1]):
        parts.append(str(row[col_b_q1]).strip())
    if col_b_q2 and pd.notna(row[col_b_q2]):
        parts.append(str(row[col_b_q2]).strip())

    if not parts and likert_cols:
        likert_text = likert_to_text(row, likert_cols)
        if likert_text:
            parts.append(likert_text)

    combined = ' '.join(parts).strip()
    return re.sub(r'\s+', ' ', combined)


df = pd.DataFrame()
df['text'] = raw_df.apply(extract_row_text, axis=1)
df['participant_id'] = [f'P{i+1:03d}' for i in range(len(df))]

ts_col = [c for c in raw_df.columns if 'timestamp' in c.lower()]
if ts_col:
    df['timestamp'] = pd.to_datetime(raw_df[ts_col[0]], errors='coerce')
    print(f'   ✅ Timestamp detected')

df = df[df['text'].str.len() > 10].reset_index(drop=True)
df['word_count'] = df['text'].apply(lambda x: len(x.split()))
n = len(df)

print(f'\n✅ {n} valid participants extracted')
print(
    f'   Word count: min={df["word_count"].min()}, max={df["word_count"].max()}, '
    f'mean={df["word_count"].mean():.0f}'
)
print('\n📋 Sample responses:')
for _, row in df.head(4).iterrows():
    print(f'  {row["participant_id"]}: {row["text"][:100]}...')


In [ ]:
from transformers import pipeline
import torch

print('🤖 Loading zero-shot classifier...')
zs_classifier = pipeline(
    'zero-shot-classification',
    model='facebook/bart-large-mnli',
    device=0 if torch.cuda.is_available() else -1,
)
print('   ✅ facebook/bart-large-mnli loaded')

SCHEMA_LABELS = [
    'addiction',
    'adhd',
    'alcoholism',
    'anxiety',
    'autism',
    'bipolarreddit',
    'bpd',
    'depression',
    'healthanxiety',
    'lonely',
    'mentalhealth',
    'ptsd',
    'schizophrenia',
    'socialanxiety',
    'suicidewatch',
]


ZS_LABELS = [
    'feeling depressed, empty, or persistently sad',
    'anxiety, chronic worry, overthinking, or excessive stress',
    'suicidal thoughts, wanting to die, or emotional crisis',
    'ADHD, attention difficulties, or difficulty focusing',
    'bipolar disorder, extreme mood swings, or emotional highs and lows',
    'borderline personality, emotional instability, or fear of abandonment',
    'PTSD, unresolved trauma, or past abuse',
    'social anxiety or fear of social situations',
    'health anxiety or excessive worry about physical illness',
    'loneliness, isolation, or feeling disconnected from others',
    'schizophrenia, paranoia, or loss of contact with reality',
    'alcohol misuse or alcohol dependency',
    'addiction or substance dependency',
    'autism spectrum or difficulty with social understanding',
    'general mental health distress or emotional difficulty',
]
ZS_TO_SCHEMA = {
    'feeling depressed, empty, or persistently sad': 'depression',
    'anxiety, chronic worry, overthinking, or excessive stress': 'anxiety',
    'suicidal thoughts, wanting to die, or emotional crisis': 'suicidewatch',
    'ADHD, attention difficulties, or difficulty focusing': 'adhd',
    'bipolar disorder, extreme mood swings, or emotional highs and lows': 'bipolarreddit',
    'borderline personality, emotional instability, or fear of abandonment': 'bpd',
    'PTSD, unresolved trauma, or past abuse': 'ptsd',
    'social anxiety or fear of social situations': 'socialanxiety',
    'health anxiety or excessive worry about physical illness': 'healthanxiety',
    'loneliness, isolation, or feeling disconnected from others': 'lonely',
    'schizophrenia, paranoia, or loss of contact with reality': 'schizophrenia',
    'alcohol misuse or alcohol dependency': 'alcoholism',
    'addiction or substance dependency': 'addiction',
    'autism spectrum or difficulty with social understanding': 'autism',
    'general mental health distress or emotional difficulty': 'mentalhealth',
}


REDDIT_FREQ = {
    'depression': 0.30,
    'suicidewatch': 0.12,
    'mentalhealth': 0.10,
    'anxiety': 0.07,
    'adhd': 0.06,
    'lonely': 0.06,
    'socialanxiety': 0.05,
    'bipolarreddit': 0.05,
    'bpd': 0.04,
    'ptsd': 0.04,
    'healthanxiety': 0.03,
    'autism': 0.03,
    'schizophrenia': 0.02,
    'alcoholism': 0.02,
    'addiction': 0.01,
}
UNIFORM = 1.0 / len(le.classes_)
CORRECTION = np.array([UNIFORM / REDDIT_FREQ.get(c, UNIFORM) for c in le.classes_])


CRISIS_WORDS = {
    'suicide',
    'suicidal',
    'die',
    'dying',
    'death',
    'dead',
    'kill',
    'end my life',
    'want to die',
    'better off dead',
    'beauty of death',
    'live or die',
    'should i live',
    'no reason to live',
    'cant go on',
    'harm myself',
    'hurt myself',
    'self harm',
}
DISTRESS_WORDS = CRISIS_WORDS | {
    'worthless',
    'hopeless',
    'empty',
    'numb',
    'hurt',
    'useless',
    'meaningless',
    'depressed',
    'anxiety',
    'panic',
    'trauma',
    'abuse',
    'exhausted',
    'stuck',
    'lonely',
    'trapped',
    'cry',
    'crying',
    'dark',
    'overwhelmed',
    'breakdown',
    'relapse',
    'scared',
    'fear',
    'helpless',
    'heavy',
    'burden',
    'suffocate',
}


def run_your_model(texts):
    X = tfidf.transform(texts)
    probas = []
    for model in models.values():
        if hasattr(model, 'predict_proba'):
            p = model.predict_proba(X)
        else:
            d = model.decision_function(X)
            p = np.exp(d - d.max(axis=1, keepdims=True))
            p /= p.sum(axis=1, keepdims=True)
        probas.append(p)
    avg = np.stack(probas).mean(axis=0) * CORRECTION
    return avg / avg.sum(axis=1, keepdims=True)


def run_zeroshot(text):
    result = zs_classifier(text[:1024], candidate_labels=ZS_LABELS, multi_label=False)
    schema = {l: 0.0 for l in SCHEMA_LABELS}
    for lbl, score in zip(result['labels'], result['scores']):
        schema[ZS_TO_SCHEMA.get(lbl, 'mentalhealth')] += float(score)
    total = sum(schema.values()) or 1.0
    return {k: v / total for k, v in schema.items()}


def schema_to_idx(label):
    classes = list(le.classes_)
    return classes.index(label) if label in classes else None


def contains_crisis(text):
    """True if ANY crisis keyword appears in the text (case-insensitive)."""
    txt = text.lower()
    return any(kw in txt for kw in CRISIS_WORDS)


def classify_batch(texts, threshold=CONFIDENCE_THRESHOLD):
    your_proba = run_your_model(texts)
    rows, blended_rows = [], []
    print(f'\n   Classifying {len(texts)} participants...')

    for i, text in enumerate(texts):
        your_dict = {
            le.classes_[j]: float(your_proba[i][j]) for j in range(len(le.classes_))
        }
        zs_dict = run_zeroshot(text)

        blended = {
            lbl: REDDIT_WEIGHT * your_dict.get(lbl, 0.0)
            + BART_WEIGHT * zs_dict.get(lbl, 0.0)
            for lbl in SCHEMA_LABELS
        }

        text_lower = text.lower()
        if contains_crisis(text):

            blended['suicidewatch'] = max(blended.get('suicidewatch', 0), 0.50)
            total = sum(blended.values())
            blended = {k: v / total for k, v in blended.items()}

        elif (
            zs_dict.get('suicidewatch', 0) > 0.15
            or your_dict.get('suicidewatch', 0) > 0.20
        ):
            blended['suicidewatch'] = max(
                blended.get('suicidewatch', 0),
                zs_dict.get('suicidewatch', 0),
                your_dict.get('suicidewatch', 0),
            )
            total = sum(blended.values())
            blended = {k: v / total for k, v in blended.items()}

        text_words = set(re.sub(r'[^a-z ]', '', text_lower).split())
        if not (text_words & DISTRESS_WORDS):
            blended['mentalhealth'] = blended.get('mentalhealth', 0) * 2.0
            total = sum(blended.values())
            blended = {k: v / total for k, v in blended.items()}

        final_label = max(blended, key=blended.get)
        final_conf = blended[final_label]
        your_top = max(your_dict, key=your_dict.get)
        zs_top = max(zs_dict, key=zs_dict.get)
        agree = your_top == zs_top == final_label

        if final_label == 'suicidewatch':
            risk_level, status = 'high', 'condition_detected'
        elif final_conf >= threshold:
            risk_level = 'high' if agree else 'borderline'
            status = 'condition_detected' if agree else 'borderline'
        elif final_conf >= BORDERLINE_LOWER:
            risk_level, status = 'borderline', 'borderline'
        else:
            final_label = 'mentalhealth'
            risk_level, status = 'low', 'normal'

        top5 = sorted(blended.items(), key=lambda x: -x[1])[:5]
        rows.append(
            {
                'participant_id': df['participant_id'].iloc[i],
                'classification': final_label,
                'confidence': round(final_conf, 4),
                'status': status,
                'risk_level': risk_level,
                'models_agree': agree,
                'your_model_top': your_top,
                'zeroshot_top': zs_top,
                'top1_condition': top5[0][0],
                'top1_prob': round(top5[0][1], 4),
                'top2_condition': top5[1][0] if len(top5) > 1 else '',
                'top2_prob': round(top5[1][1], 4) if len(top5) > 1 else 0.0,
                'word_count': df['word_count'].iloc[i],
            }
        )
        blended_full = {l: blended.get(l, 0.0) for l in SCHEMA_LABELS}
        blended_full['participant_id'] = df['participant_id'].iloc[i]
        blended_rows.append(blended_full)

        if (i + 1) % 5 == 0 or (i + 1) == len(texts):
            print(f'   ... {i+1}/{len(texts)} done')

    global blended_matrix_df
    blended_matrix_df = pd.DataFrame(blended_rows).set_index('participant_id')
    return pd.DataFrame(rows)


print('\n✅ Ensemble functions ready — run next cell to classify')


In [ ]:
print('🚀 Running Ensemble Classification...')
print('=' * 70)

texts_list = df['text'].tolist()
results_df = classify_batch(texts_list)

n_normal = (results_df['risk_level'] == 'low').sum()
n_borderline = (results_df['risk_level'] == 'borderline').sum()
n_condition = (results_df['risk_level'] == 'high').sum()
n_agree = results_df['models_agree'].sum()

print(f'\n✅ Done — {n} participants classified')
print(f'   🟢 Normal            : {n_normal}  ({n_normal/n*100:.1f}%)')
print(f'   🟡 Borderline        : {n_borderline}  ({n_borderline/n*100:.1f}%)')
print(f'   🔴 Condition detected: {n_condition}  ({n_condition/n*100:.1f}%)')
print(f'   🤝 Both agreed       : {n_agree}  ({n_agree/n*100:.1f}%)')
print('\n📊 Conditions found:')
cond_counts = results_df['classification'].value_counts()
max_cnt = cond_counts.max()
for cond, cnt in cond_counts.items():
    bar = '█' * int(cnt / max_cnt * 20)
    print(f'   {cond:<20} : {cnt:>3} ({cnt/n*100:.1f}%)  {bar}')


In [ ]:
print('📊 Chart 1 — Trend Analysis Dashboard')

fig = plt.figure(figsize=(20, 15))
fig.suptitle(
    f'Mental Health Trend Analysis  —  N={n} Participants',
    fontsize=17,
    fontweight='bold',
    y=0.98,
)
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.50, wspace=0.38)
cond_counts_plot = results_df['classification'].value_counts()


ax1 = fig.add_subplot(gs[0, 0])
pie_colors = [CLASS_COLOR.get(c, '#aaa') for c in cond_counts_plot.index]
wedges, _, autotexts = ax1.pie(
    cond_counts_plot.values,
    labels=cond_counts_plot.index,
    autopct='%1.1f%%',
    colors=pie_colors,
    startangle=140,
    textprops={'fontsize': 8},
    pctdistance=0.80,
)
for at in autotexts:
    at.set_fontsize(7)
ax1.set_title('Condition Distribution', fontweight='bold', fontsize=12)


ax2 = fig.add_subplot(gs[0, 1])
risk_order = ['low', 'borderline', 'high']
risk_labels = {'low': 'Normal', 'borderline': 'Borderline', 'high': 'Detected'}
risk_colors = {'low': '#2ecc71', 'borderline': '#f39c12', 'high': '#e74c3c'}
risk_counts = results_df['risk_level'].value_counts()
present = [r for r in risk_order if r in risk_counts.index]
vals = [risk_counts[r] for r in present]
bars = ax2.bar(
    [risk_labels[r] for r in present],
    vals,
    color=[risk_colors[r] for r in present],
    edgecolor='white',
    width=0.5,
)
for bar, val in zip(bars, vals):
    ax2.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.2,
        f'{val}\n({val/n*100:.0f}%)',
        ha='center',
        va='bottom',
        fontsize=11,
        fontweight='bold',
    )
ax2.set_ylabel('Participants')
ax2.set_ylim(0, max(vals) * 1.35)
ax2.set_title('Mental Health Risk Level', fontweight='bold', fontsize=12)


ax3 = fig.add_subplot(gs[0, 2])
top_conds = cond_counts_plot.head(10)
bars3 = ax3.barh(
    top_conds.index[::-1],
    top_conds.values[::-1],
    color=[CLASS_COLOR.get(c, '#aaa') for c in top_conds.index[::-1]],
    edgecolor='white',
)
for bar, val in zip(bars3, top_conds.values[::-1]):
    ax3.text(
        bar.get_width() + 0.05,
        bar.get_y() + bar.get_height() / 2,
        f'{val}',
        va='center',
        fontsize=9,
        fontweight='bold',
    )
ax3.set_xlabel('Count')
ax3.set_xlim(0, top_conds.max() * 1.3)
ax3.set_title('Top Detected Conditions', fontweight='bold', fontsize=12)


ax4 = fig.add_subplot(gs[1, 0])
conf_vals = results_df['confidence']
ax4.hist(conf_vals, bins=15, color='#3498db', alpha=0.75, edgecolor='white')
ax4.axvspan(0, BORDERLINE_LOWER, alpha=0.10, color='#2ecc71', label='Normal')
ax4.axvspan(
    BORDERLINE_LOWER,
    CONFIDENCE_THRESHOLD,
    alpha=0.10,
    color='#f39c12',
    label='Borderline',
)
ax4.axvspan(CONFIDENCE_THRESHOLD, 1, alpha=0.10, color='#e74c3c', label='Condition')
ax4.axvline(
    conf_vals.mean(),
    color='#e74c3c',
    lw=2,
    linestyle='--',
    label=f'Mean={conf_vals.mean():.2f}',
)
ax4.axvline(
    conf_vals.median(),
    color='#9b59b6',
    lw=2,
    linestyle='--',
    label=f'Median={conf_vals.median():.2f}',
)
ax4.set_xlabel('Confidence Score')
ax4.set_ylabel('Frequency')
ax4.set_title('Confidence Distribution', fontweight='bold', fontsize=12)
ax4.legend(fontsize=7)


ax5 = fig.add_subplot(gs[1, 1:])
conditions_present = results_df['classification'].unique().tolist()
box_data = [
    results_df[results_df['classification'] == c]['confidence'].values
    for c in conditions_present
]
bp = ax5.boxplot(
    box_data, patch_artist=True, medianprops=dict(color='black', linewidth=2)
)
for patch, cname in zip(bp['boxes'], conditions_present):
    patch.set_facecolor(CLASS_COLOR.get(cname, '#aaa'))
    patch.set_alpha(0.75)
ax5.set_xticks(range(1, len(conditions_present) + 1))
ax5.set_xticklabels(conditions_present, rotation=35, ha='right', fontsize=8)
ax5.set_ylabel('Confidence Score')
ax5.set_title('Confidence by Condition', fontweight='bold', fontsize=12)
ax5.axhline(
    CONFIDENCE_THRESHOLD,
    color='#e74c3c',
    lw=1.5,
    linestyle='--',
    label=f'Thr={CONFIDENCE_THRESHOLD}',
)
ax5.axhline(
    BORDERLINE_LOWER,
    color='#f39c12',
    lw=1.5,
    linestyle='--',
    label=f'Border={BORDERLINE_LOWER}',
)
ax5.legend(fontsize=8)


ax6 = fig.add_subplot(gs[2, :])
heat_matrix = blended_matrix_df.values.T
im = ax6.imshow(heat_matrix, aspect='auto', cmap='RdYlGn', vmin=0, vmax=0.6)
risk_icon = {'low': 'N', 'borderline': 'B', 'high': '!'}
for xi, pid in enumerate(blended_matrix_df.index):
    icon = risk_icon[
        results_df.loc[results_df['participant_id'] == pid, 'risk_level'].values[0]
    ]
    ax6.text(xi, -0.9, icon, ha='center', va='top', fontsize=6.5, fontweight='bold')
ax6.set_xticks(range(n))
ax6.set_xticklabels(blended_matrix_df.index, rotation=90, fontsize=6)
ax6.set_yticks(range(len(blended_matrix_df.columns)))
ax6.set_yticklabels(blended_matrix_df.columns, fontsize=8)
ax6.set_title(
    'Blended Probability Heatmap  (N=Normal · B=Borderline · !=Detected)',
    fontweight='bold',
    fontsize=11,
)
plt.colorbar(im, ax=ax6, label='Blended Probability', fraction=0.015)

plt.savefig(
    _paths.output_path('chart1_trend_dashboard.png'), dpi=150, bbox_inches='tight'
)
plt.show()
print('✅ Saved → chart1_trend_dashboard.png')


In [ ]:
print('📊 Chart 2 — Response Pattern Analysis')
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('Response Pattern Analysis', fontsize=14, fontweight='bold')

ax = axes[0]
ax.hist(
    results_df['word_count'], bins=12, color='#9b59b6', alpha=0.75, edgecolor='white'
)
ax.axvline(
    results_df['word_count'].mean(),
    color='#e74c3c',
    lw=2,
    linestyle='--',
    label=f'Mean={results_df["word_count"].mean():.0f} words',
)
ax.set_xlabel('Word Count')
ax.set_ylabel('Participants')
ax.set_title('Response Length Distribution', fontweight='bold')
ax.legend()

ax = axes[1]
order = (
    results_df.groupby('classification')['word_count']
    .mean()
    .sort_values(ascending=False)
    .index
)
sns.boxplot(
    data=results_df,
    x='classification',
    y='word_count',
    order=order,
    palette={c: CLASS_COLOR.get(c, '#aaa') for c in order},
    ax=ax,
)
ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha='right', fontsize=8)
ax.set_xlabel('')
ax.set_ylabel('Word Count')
ax.set_title('Word Count by Condition', fontweight='bold')

ax = axes[2]
agree_counts = results_df['models_agree'].value_counts()
agree_labels = {True: f'Both agree\n{n_agree}/{n}', False: f'Differ\n{n-n_agree}/{n}'}
ax.pie(
    agree_counts.values,
    labels=[agree_labels.get(k, str(k)) for k in agree_counts.index],
    autopct='%1.0f%%',
    colors=['#2ecc71', '#e74c3c'][: len(agree_counts)],
    startangle=90,
    textprops={'fontsize': 10},
)
ax.set_title('Model Agreement\n(Your model vs BART)', fontweight='bold')

plt.tight_layout()
plt.savefig(
    _paths.output_path('chart2_response_patterns.png'), dpi=150, bbox_inches='tight'
)
plt.show()
print('✅ Saved → chart2_response_patterns.png')


In [ ]:
print('📊 Chart 3 — Model Agreement & Confidence Deep-Dive')
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('Model Agreement & Confidence Analysis', fontsize=14, fontweight='bold')

ax = axes[0]
agree_by_cond = (
    results_df.groupby('classification')['models_agree']
    .mean()
    .sort_values(ascending=False)
)
bars_a = ax.bar(
    agree_by_cond.index,
    agree_by_cond.values * 100,
    color=[CLASS_COLOR.get(c, '#aaa') for c in agree_by_cond.index],
    edgecolor='white',
)
ax.set_xticklabels(agree_by_cond.index, rotation=40, ha='right', fontsize=8)
ax.set_ylabel('Agreement Rate (%)')
ax.set_ylim(0, 115)
ax.set_title('Model Agreement Rate by Condition', fontweight='bold')
ax.axhline(50, color='gray', lw=1, linestyle='--')
for bar, val in zip(bars_a, agree_by_cond.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        f'{val*100:.0f}%',
        ha='center',
        va='bottom',
        fontsize=8,
    )

ax = axes[1]
risk_palette = {'low': '#2ecc71', 'borderline': '#f39c12', 'high': '#e74c3c'}
order_r = [
    r for r in ['low', 'borderline', 'high'] if r in results_df['risk_level'].unique()
]
sns.violinplot(
    data=results_df,
    x='risk_level',
    y='confidence',
    order=order_r,
    palette=risk_palette,
    inner='quartile',
    ax=ax,
)
ax.set_xticklabels(['Normal', 'Borderline', 'Detected'][: len(order_r)])
ax.set_xlabel('Risk Level')
ax.set_ylabel('Confidence Score')
ax.set_title('Confidence by Risk Level', fontweight='bold')
ax.axhline(CONFIDENCE_THRESHOLD, color='#e74c3c', lw=1.5, linestyle='--')
ax.axhline(BORDERLINE_LOWER, color='#f39c12', lw=1.5, linestyle='--')

ax = axes[2]
scatter_colors = [CLASS_COLOR.get(c, '#aaa') for c in results_df['classification']]
ax.scatter(
    results_df['word_count'],
    results_df['confidence'],
    c=scatter_colors,
    alpha=0.75,
    s=60,
    edgecolors='white',
    lw=0.5,
)
ax.axhline(CONFIDENCE_THRESHOLD, color='#e74c3c', lw=1.5, linestyle='--')
ax.axhline(BORDERLINE_LOWER, color='#f39c12', lw=1.5, linestyle='--')
ax.set_xlabel('Response Word Count')
ax.set_ylabel('Confidence Score')
ax.set_title('Word Count vs Confidence', fontweight='bold')
handles_sc = [
    mpatches.Patch(color=CLASS_COLOR.get(c, '#aaa'), label=c)
    for c in results_df['classification'].unique()
]
ax.legend(handles=handles_sc, fontsize=6.5, ncol=2)

plt.tight_layout()
plt.savefig(
    _paths.output_path('chart3_agreement_confidence.png'), dpi=150, bbox_inches='tight'
)
plt.show()
print('✅ Saved → chart3_agreement_confidence.png')


In [ ]:
print('📊 Chart 4 — Condition Co-occurrence & Model Comparison')
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle(
    'Condition Co-occurrence & Model Comparison', fontsize=14, fontweight='bold'
)

ax = axes[0]
top2_df = results_df[results_df['top2_condition'] != '']
if len(top2_df) > 0:
    co_matrix = pd.crosstab(top2_df['top1_condition'], top2_df['top2_condition'])
    co_matrix = co_matrix.reindex(
        index=[c for c in SCHEMA_LABELS if c in co_matrix.index],
        columns=[c for c in SCHEMA_LABELS if c in co_matrix.columns],
        fill_value=0,
    )
    sns.heatmap(
        co_matrix,
        annot=True,
        fmt='d',
        cmap='Blues',
        linewidths=0.5,
        ax=ax,
        cbar_kws={'label': 'Count'},
    )
    ax.set_xlabel('2nd Condition')
    ax.set_ylabel('1st Condition')
    ax.set_title('Condition Co-occurrence\n(Top-1 vs Top-2)', fontweight='bold')

ax = axes[1]
your_counts = results_df['your_model_top'].value_counts()
zs_counts = results_df['zeroshot_top'].value_counts()
all_conds = sorted(set(list(your_counts.index) + list(zs_counts.index)))
x = np.arange(len(all_conds))
w = 0.35
ax.bar(
    x - w / 2,
    [your_counts.get(c, 0) for c in all_conds],
    w,
    label='Your Model (LR+SVM+RF)',
    color='#3498db',
    alpha=0.85,
    edgecolor='white',
)
ax.bar(
    x + w / 2,
    [zs_counts.get(c, 0) for c in all_conds],
    w,
    label='BART Zero-Shot',
    color='#e74c3c',
    alpha=0.85,
    edgecolor='white',
)
ax.set_xticks(x)
ax.set_xticklabels(all_conds, rotation=40, ha='right', fontsize=8)
ax.set_ylabel('Participants')
ax.set_title('Your Model vs BART Top Prediction', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig(_paths.output_path('chart4_cooccurrence.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved → chart4_cooccurrence.png')


In [ ]:
print('📊 Chart 5 — Condition Probability Profiles')
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Blended Probability Profiles', fontsize=14, fontweight='bold')

ax = axes[0]
mean_probs = blended_matrix_df.mean(axis=0).sort_values(ascending=False)
bars5 = ax.bar(
    mean_probs.index,
    mean_probs.values,
    color=[CLASS_COLOR.get(c, '#aaa') for c in mean_probs.index],
    edgecolor='white',
)
ax.set_xticklabels(mean_probs.index, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Mean Blended Probability')
ax.set_title('Average Probability per Condition\n(all participants)', fontweight='bold')
for bar, val in zip(bars5, mean_probs.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.002,
        f'{val:.3f}',
        ha='center',
        va='bottom',
        fontsize=7,
    )

ax = axes[1]
ax.remove()
ax_r = fig.add_subplot(1, 2, 2, projection='polar')
categories = SCHEMA_LABELS
N = len(categories)
angles = [i / N * 2 * np.pi for i in range(N)] + [0]
mean_vals = [blended_matrix_df[c].mean() for c in categories] + [
    blended_matrix_df[categories[0]].mean()
]
ax_r.set_theta_offset(np.pi / 2)
ax_r.set_theta_direction(-1)
ax_r.set_xticks(angles[:-1])
ax_r.set_xticklabels(categories, fontsize=7)
ax_r.plot(angles, mean_vals, 'o-', lw=2, color='#e74c3c')
ax_r.fill(angles, mean_vals, alpha=0.25, color='#e74c3c')
ax_r.set_title('Radar: Group Average\nCondition Profile', fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig(_paths.output_path('chart5_profiles.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved → chart5_profiles.png')


In [ ]:
print('📊 Chart 6 — Submission Timeline')
if 'timestamp' in df.columns and df['timestamp'].notna().any():
    ts_results = results_df.copy()
    ts_results['timestamp'] = df['timestamp'].values
    ts_results = ts_results.dropna(subset=['timestamp']).sort_values('timestamp')
    fig, axes = plt.subplots(1, 2, figsize=(18, 5))
    fig.suptitle('Submission Timeline Analysis', fontsize=14, fontweight='bold')
    ax = axes[0]
    risk_num = {'low': 0, 'borderline': 1, 'high': 2}
    rc = {
        r: c
        for r, c in [('low', '#2ecc71'), ('borderline', '#f39c12'), ('high', '#e74c3c')]
    }
    ax.scatter(
        ts_results['timestamp'],
        ts_results['risk_level'].map(risk_num),
        c=[rc[r] for r in ts_results['risk_level']],
        s=80,
        alpha=0.8,
        edgecolors='white',
        zorder=3,
    )
    ax.set_yticks([0, 1, 2])
    ax.set_yticklabels(['Normal', 'Borderline', 'Detected'])
    ax.set_xlabel('Submission Time')
    ax.set_title('Risk Level Over Time', fontweight='bold')
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
    ax = axes[1]
    ts_results['date'] = ts_results['timestamp'].dt.date
    daily = ts_results.groupby('date').size()
    ax.bar(
        range(len(daily)), daily.values, color='#3498db', edgecolor='white', alpha=0.8
    )
    ax.set_xticks(range(len(daily)))
    ax.set_xticklabels(
        [str(d) for d in daily.index], rotation=30, ha='right', fontsize=8
    )
    ax.set_ylabel('Responses')
    ax.set_title('Submissions per Day', fontweight='bold')
    plt.tight_layout()
    plt.savefig(_paths.output_path('chart6_timeline.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Saved → chart6_timeline.png')
else:
    print('ℹ️  No timestamp column — skipping timeline chart.')


In [ ]:
print('🔬 Chart 7 — SHAP Global XAI')
print('=' * 70)

X_shap = tfidf.transform(texts_list)
feature_names = np.array(tfidf.get_feature_names_out())

print('⚙️  Computing SHAP values...')
shap_explainer = shap.LinearExplainer(lr, X_shap, feature_perturbation='interventional')
shap_vals = shap_explainer.shap_values(X_shap)
print(f'   ✅ Done — {X_shap.shape[0]} participants × {len(shap_vals)} classes')

detected_schema = (
    results_df[results_df['status'] == 'condition_detected']['classification']
    .unique()
    .tolist()
)
if not detected_schema:
    detected_schema = results_df['classification'].value_counts().head(5).index.tolist()

condition_indices = [
    i for i in [schema_to_idx(s) for s in detected_schema] if i is not None
]
if not condition_indices:
    condition_indices = list(range(min(5, len(shap_vals))))

mean_abs_shap = np.mean(
    [np.abs(shap_vals[i]).mean(axis=0) for i in condition_indices], axis=0
)
top20_idx = np.argsort(mean_abs_shap)[::-1][:20]
top20_words = feature_names[top20_idx]
top20_vals = mean_abs_shap[top20_idx]

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
fig.suptitle('SHAP Global XAI — Most Influential Words', fontsize=14, fontweight='bold')

ax = axes[0]
colors_shap = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, 20))
ax.barh(top20_words[::-1], top20_vals[::-1], color=colors_shap[::-1])
ax.set_xlabel('Mean |SHAP Value|')
ax.set_title(
    'Top 20 Influential Words\n(across detected conditions)', fontweight='bold'
)

ax = axes[1]
top_classes = detected_schema[:6]
plot_data = {}
for cname in top_classes:
    cidx = schema_to_idx(cname)
    if cidx is None:
        continue
    mean_abs = np.abs(shap_vals[cidx]).mean(axis=0)
    for fi in np.argsort(mean_abs)[::-1][:3]:
        plot_data[f'{feature_names[fi]}\n({cname})'] = (mean_abs[fi], cname)

if plot_data:
    sorted_pd = sorted(plot_data.items(), key=lambda x: -x[1][0])
    words_pd = [s[0] for s in sorted_pd]
    vals_pd = [s[1][0] for s in sorted_pd]
    cnames_pd = [s[1][1] for s in sorted_pd]
    ax.barh(
        words_pd,
        vals_pd,
        color=[CLASS_COLOR.get(c, '#aaa') for c in cnames_pd],
        alpha=0.85,
        edgecolor='white',
    )
    ax.set_xlabel('Mean |SHAP Value|')
    ax.set_title('Top 3 Key Words per Condition', fontweight='bold')
    ax.legend(
        handles=[
            mpatches.Patch(color=CLASS_COLOR.get(c, '#aaa'), label=c)
            for c in top_classes
        ]
    )

plt.tight_layout()
plt.savefig(_paths.output_path('chart7_shap_global.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved → chart7_shap_global.png')


In [ ]:
print('🔬 Chart 8 — SHAP Per-Condition Deep Dive')

detected_conditions = (
    results_df[results_df['status'] == 'condition_detected']['classification']
    .value_counts()
    .head(4)
    .index.tolist()
)
if not detected_conditions:
    detected_conditions = (
        results_df['top1_condition'].value_counts().head(3).index.tolist()
    )

n_plots = len(detected_conditions)
if n_plots == 0:
    print('⚠️  No detected conditions.')
else:
    fig, axes = plt.subplots(1, n_plots, figsize=(6 * n_plots, 8), squeeze=False)
    axes = axes[0]
    fig.suptitle(
        'SHAP Per-Condition: Words That Drive Each Classification',
        fontsize=13,
        fontweight='bold',
    )

    for ax, cname in zip(axes, detected_conditions):
        cidx = schema_to_idx(cname)
        if cidx is None:
            ax.text(0.5, 0.5, f'{cname}\nnot in encoder', ha='center', va='center')
            ax.axis('off')
            continue

        sv = shap_vals[cidx]
        pos_mean = np.where(sv > 0, sv, 0).mean(axis=0)
        neg_mean = np.where(sv < 0, sv, 0).mean(axis=0)
        combined_idx = list(
            dict.fromkeys(
                list(np.argsort(pos_mean)[::-1][:8]) + list(np.argsort(neg_mean)[:8])
            )
        )[:12]
        words_plot = feature_names[combined_idx]
        vals_plot = sv.mean(axis=0)[combined_idx]
        sorted_pairs = sorted(zip(vals_plot, words_plot))
        vals_plot, words_plot = zip(*sorted_pairs)
        ax.barh(
            words_plot,
            vals_plot,
            color=['#e74c3c' if v > 0 else '#3498db' for v in vals_plot],
            alpha=0.85,
            edgecolor='white',
        )
        ax.axvline(0, color='black', lw=1)
        ax.set_title(f'[{cname.upper()}]', fontweight='bold', fontsize=11)
        ax.set_xlabel('Mean SHAP Value', fontsize=9)
        ax.legend(
            handles=[
                mpatches.Patch(color='#e74c3c', label='Pushes TOWARD'),
                mpatches.Patch(color='#3498db', label='Pushes AWAY'),
            ],
            fontsize=7,
        )

    plt.tight_layout()
    plt.savefig(
        _paths.output_path('chart8_shap_per_condition.png'),
        dpi=150,
        bbox_inches='tight',
    )
    plt.show()
    print('✅ Saved → chart8_shap_per_condition.png')

    print('\n📋 SEMANTIC SUMMARY')
    print('=' * 70)
    for cname in detected_conditions:
        cidx = schema_to_idx(cname)
        if cidx is None:
            continue
        pos_mean = np.where(shap_vals[cidx] > 0, shap_vals[cidx], 0).mean(axis=0)
        top_words = feature_names[np.argsort(pos_mean)[::-1][:8]]
        n_in_cond = (results_df['classification'] == cname).sum()
        print(f'\n  [{cname.upper()}]  ({n_in_cond} participant(s))')
        print(f'     Key words: {", ".join(top_words)}')


In [ ]:
print('👤 Individual Participant Results')
print('=' * 70)

display_df = results_df[
    [
        'participant_id',
        'classification',
        'confidence',
        'status',
        'top1_condition',
        'top1_prob',
        'top2_condition',
        'top2_prob',
        'models_agree',
        'your_model_top',
        'zeroshot_top',
        'word_count',
    ]
].copy()
display_df['confidence'] = display_df['confidence'].apply(lambda x: f'{x:.1%}')
display_df['top1_prob'] = display_df['top1_prob'].apply(lambda x: f'{x:.1%}')
display_df['top2_prob'] = display_df['top2_prob'].apply(lambda x: f'{x:.1%}')

print(
    f'{"ID":<8} {"Predicted":<20} {"Conf":<8} {"YourModel":<18} {"BART":<18} {"Agree":<7} Status'
)
print('-' * 95)
for _, row in results_df.iterrows():
    icon = (
        'N'
        if row['status'] == 'normal'
        else ('B' if row['status'] == 'borderline' else '!')
    )
    print(
        f"{row['participant_id']:<8} {row['classification']:<20} {row['confidence']:.0%}    "
        f"{row['your_model_top']:<18} {row['zeroshot_top']:<18} {str(row['models_agree']):<7} [{icon}] {row['status']}"
    )

try:
    from IPython.display import display as ipy_display

    def status_color(val):
        return {
            'normal': 'background-color:#d5f5e3',
            'borderline': 'background-color:#fef9e7',
        }.get(val, 'background-color:#fadbd8')

    ipy_display(
        display_df.style.applymap(status_color, subset=['status'])
        .set_properties(**{'font-size': '11px'})
        .set_caption(f'Ensemble Classification — {n} Participants')
    )
except Exception:
    print(display_df.to_string())


In [ ]:
print('💾 Exporting results...')
results_df.to_csv(_paths.output_path('mental_health_results_full.csv'), index=False)
cond_dist = results_df['classification'].value_counts().reset_index()
cond_dist.columns = ['condition', 'count']
cond_dist['percentage'] = (cond_dist['count'] / n * 100).round(1)
cond_dist.to_csv(_paths.output_path('mental_health_distribution.csv'), index=False)
blended_matrix_df.to_csv(_paths.output_path('mental_health_blended_proba.csv'))
summary = {
    'total_participants': [n],
    'normal_count': [n_normal],
    'borderline_count': [n_borderline],
    'condition_detected_count': [n_condition],
    'normal_pct': [f'{n_normal/n*100:.1f}%'],
    'condition_pct': [f'{n_condition/n*100:.1f}%'],
    'mean_confidence': [f'{results_df["confidence"].mean():.1%}'],
    'models_agreed_pct': [f'{n_agree/n*100:.1f}%'],
    'confidence_threshold': [CONFIDENCE_THRESHOLD],
    'borderline_threshold': [BORDERLINE_LOWER],
    'reddit_bart_weights': [f'{REDDIT_WEIGHT:.0%} / {BART_WEIGHT:.0%}'],
    'schema': [' | '.join(SCHEMA_LABELS)],
}
pd.DataFrame(summary).T.reset_index().rename(
    columns={'index': 'metric', 0: 'value'}
).to_csv(_paths.output_path('mental_health_summary.csv'), index=False)
pd.concat(
    [raw_df.reset_index(drop=True), results_df.reset_index(drop=True)], axis=1
).to_csv(_paths.output_path('mental_health_merged.csv'), index=False)
print('   ✅ mental_health_results_full.csv')
print('   ✅ mental_health_distribution.csv')
print('   ✅ mental_health_blended_proba.csv')
print('   ✅ mental_health_summary.csv')
print('   ✅ mental_health_merged.csv')
print('\n⚠️  Research Disclaimer: screening tool only — not a clinical diagnosis.')


## 📄 Research Paper — Methods Snippet

> *Text responses were collected via a Google Form survey and analysed using a two-model ensemble. 
A Reddit-trained TF-IDF ensemble (Logistic Regression, Linear SVM, Random Forest) provided 
clinical-vocabulary signal with class-prior bias correction. 
A zero-shot NLI classifier (facebook/bart-large-mnli; Lewis et al., 2020) provided domain-agnostic 
semantic classification using 15 survey-calibrated candidate labels mapped directly to the Reddit 
mental health taxonomy. Final probabilities were determined by a weighted blend (20% TF-IDF, 80% NLI), 
with a keyword-based crisis escalation override for suicidal ideation and a distress-vocabulary-based 
confidence adjustment. Classification thresholds were calibrated to survey language (condition: 0.20, 
borderline: 0.12). SHAP LinearExplainer (Lundberg & Lee, 2017) was applied to the Logistic Regression 
for explainability. All analysis was performed in Python 3.10 using scikit-learn, HuggingFace Transformers, and SHAP.*

**References:**
- Lewis et al. (2020). BART: Denoising Sequence-to-Sequence Pre-training. *ACL 2020*.
- Lundberg & Lee (2017). A unified approach to interpreting model predictions. *NeurIPS 2017*.
- Pedregosa et al. (2011). Scikit-learn: Machine Learning in Python. *JMLR 12*.
